In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import math, random

In [2]:
# 1. Toy Dataset

text_data = """
The cat sat on the mat.
The dog chased the ball.
Translate to German: cat = Katze
Translate to German: dog = Hund
Q: Who wrote Harry Potter? A: J.K. Rowling
Q: What is the capital of France? A: Paris
"""

chars = sorted(list(set(text_data)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

data = torch.tensor(encode(text_data), dtype=torch.long)
train_data = data[:-20]
val_data = data[-20:]

In [3]:
# 2. Mini Transformer Language Model

class TinyGPT(nn.Module):
    def __init__(self, vocab_size, n_embd=64, n_head=4, n_layer=2, block_size=64):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=n_embd, nhead=n_head),
            num_layers=n_layer
        )
        self.ln = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size)
        self.block_size = block_size

    def forward(self, idx):
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device).unsqueeze(0)
        x = self.token_emb(idx) + self.pos_emb(pos)
        x = self.blocks(x)
        x = self.ln(x)
        logits = self.head(x)
        return logits

In [4]:
# 3. Training

def get_batch(split):
    data_split = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_split) - block_size, (1,))
    x = data_split[ix:ix+block_size].unsqueeze(0)
    y = data_split[ix+1:ix+block_size+1].unsqueeze(0)
    return x, y

block_size = 64
model = TinyGPT(vocab_size)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for step in range(500):
    xb, yb = get_batch('train')
    logits = model(xb)
    loss = loss_fn(logits.view(-1, vocab_size), yb.view(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 100 == 0:
        print(f"Step {step}: loss {loss.item():.4f}")

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


Step 0: loss 3.8536
Step 100: loss 1.3971
Step 200: loss 1.4154
Step 300: loss 1.1729
Step 400: loss 1.5370


In [25]:
# 4. Generation (Zero-shot test)

def generate(prompt, max_new_tokens=60):
    model.eval()
    idx = torch.tensor(encode(prompt), dtype=torch.long).unsqueeze(0)
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits = model(idx_cond)
        probs = torch.nn.functional.softmax(logits[:, -1, :], dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, next_id), dim=1)
    return decode(idx[0].tolist())

prompt = "Q: Who wrote Harry Potter? A:"
print(generate(prompt))

Q: Who wrote Harry Potter? A: J.K.Kateralate P
Q: can: Whermatarotowrmate Ge : tho te to 
